In [7]:
import torch
import torch.nn as nn
import numpy as np
import lab07_starter as S

In [8]:
if __name__ == "__main__":
    Xtr, ytr, Xte, yte = S.load_task()
    net = S.train_feature_net(Xtr, ytr)
    fte = S.get_features(net, Xte)

    seeds = [0, 1, 2]
    print("=== Piece 2: Plain AE vs VAE (beta=1.0) ===")
    
    for seed in seeds:
        # 1. Plain AE
        ae = S.AE(d_latent=8)
        S.train(ae, S.ae_loss, Xtr, epochs=200, seed=seed)
        ae_recon = S.reconstruction_mse(ae, Xte)
        ae_samples = ae.decode(torch.randn(len(Xte), 8)).detach()
        ae_scores = S.score_samples(ae_samples, fte, net, Xtr)
        print(f"[Plain AE | Seed {seed}] Recon MSE: {ae_recon:.4f} | FD: {ae_scores['frechet']:.2f} | Prec: {ae_scores['precision']:.3f} | Rec: {ae_scores['recall']:.3f} | NearestTrain: {ae_scores['nearest_train']:.3f}")
        
        # 2. VAE (beta=1.0)
        vae = VAE(d_latent=8)
        loss_fn_b1 = lambda m, xb, yb: vae_loss_fn(m, xb, yb, beta=1.0)
        S.train(vae, loss_fn_b1, Xtr, epochs=200, seed=seed)
        
        with torch.no_grad():
            mu_test, _ = vae.encode(Xte)
            vae_recon = ((torch.sigmoid(vae.decode(mu_test)) - Xte) ** 2).mean().item()
            vae_samples = torch.sigmoid(vae.decode(torch.randn(len(Xte), 8)))
            vae_scores = S.score_samples(vae_samples, fte, net, Xtr)
        print(f"[VAE b=1.0 | Seed {seed}] Recon Term: {vae_recon:.4f} | FD: {vae_scores['frechet']:.2f} | Prec: {vae_scores['precision']:.3f} | Rec: {vae_scores['recall']:.3f} | NearestTrain: {vae_scores['nearest_train']:.3f}\n")

=== Piece 2: Plain AE vs VAE (beta=1.0) ===
[Plain AE | Seed 0] Recon MSE: 0.0150 | FD: 84.10 | Prec: 0.354 | Rec: 0.786 | NearestTrain: 2.129
[VAE b=1.0 | Seed 0] Recon Term: 0.0264 | FD: 14.30 | Prec: 0.939 | Rec: 0.373 | NearestTrain: 1.116

[Plain AE | Seed 1] Recon MSE: 0.0141 | FD: 69.26 | Prec: 0.485 | Rec: 0.791 | NearestTrain: 2.081
[VAE b=1.0 | Seed 1] Recon Term: 0.0263 | FD: 12.02 | Prec: 0.933 | Rec: 0.521 | NearestTrain: 1.154

[Plain AE | Seed 2] Recon MSE: 0.0146 | FD: 71.35 | Prec: 0.370 | Rec: 0.702 | NearestTrain: 2.107
[VAE b=1.0 | Seed 2] Recon Term: 0.0269 | FD: 13.96 | Prec: 0.900 | Rec: 0.493 | NearestTrain: 1.148



In [9]:
class VAE(nn.Module):
    def __init__(self, d_in=64, d_hidden=64, d_latent=8):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(d_in, d_hidden),
            nn.ReLU()
        )
        self.fc_mu = nn.Linear(d_hidden, d_latent)
        self.fc_var = nn.Linear(d_hidden, d_latent)
        
        self.dec = nn.Sequential(
            nn.Linear(d_latent, d_hidden),
            nn.ReLU(),
            nn.Linear(d_hidden, d_in) # Returns logits for Bernoulli likelihood (BCEWithLogitsLoss)
        )

    def encode(self, x):
        h = self.enc(x)
        return self.fc_mu(h), self.fc_var(h)

    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return self.dec(z)

    def forward(self, x):
        mu, log_var = self.encode(x)
        z = self.reparameterize(mu, log_var)
        return self.decode(z), mu, log_var

def vae_loss_fn(model, xb, yb=None, beta=1.0):
    """
    Reconstruction term: Summed over pixels (BCE with logits).
    KL divergence: Summed over latent dimensions, closed-form standard normal.
    Both averaged over the batch.
    """
    recon_logits, mu, log_var = model(xb)
    
    # Reconstruction: Binary Cross Entropy with logits summed over pixels, averaged over batch
    recon_loss = nn.functional.binary_cross_entropy_with_logits(
        recon_logits, xb, reduction='none'
    ).sum(dim=1).mean()
    
    # KL Divergence: -0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2) summed over latent dims, averaged over batch
    kl_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp(), dim=1).mean()
    
    return recon_loss + beta * kl_loss

if __name__ == "__main__":
    torch.manual_seed(0)
    Xtr, ytr, Xte, yte = S.load_task()
    vae = VAE()

    # Check 1: Closed-form KL vs Monte Carlo estimate on random Gaussians
    mu = torch.randn(1000, 8)
    log_var = torch.randn(1000, 8)
    # Closed form
    cf_kl = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp(), dim=1).mean()
    # Monte Carlo estimate: E_{q}[log q(z|x) - log p(z)]
    std = torch.exp(0.5 * log_var)
    eps = torch.randn(10000, 1000, 8)
    z = mu.unsqueeze(0) + eps * std.unsqueeze(0)
    # log q(z|x)
    log_q = -0.5 * (((z - mu.unsqueeze(0)) / std.unsqueeze(0))**2 + 2 * torch.log(std.unsqueeze(0)) + np.log(2 * np.pi)).sum(dim=-1).mean()
    # log p(z)
    log_p = -0.5 * (z**2 + np.log(2 * np.pi)).sum(dim=-1).mean()
    mc_kl = log_q - log_p
    print(f"Check 1 - Closed-form KL: {cf_kl.item():.4f} | Monte Carlo KL: {mc_kl.item():.4f}")

  # Check 2: Gradient norm reaching encoder's mean layer with and without trick
    x_sample = Xtr[:4]
    
    # --- With trick ---
    vae.zero_grad()
    mu, log_var = vae.encode(x_sample)
    mu.retain_grad()  # Keep gradient for non-leaf tensor mu
    log_var.retain_grad()
    
    z_trick = vae.reparameterize(mu, log_var)
    loss_trick = vae.decode(z_trick).sum()
    loss_trick.backward()
    grad_with_trick = mu.grad.norm().item()

    # --- Without trick (detached) ---
    vae.zero_grad()
    mu_det, log_var_det = vae.encode(x_sample)
    mu_det.retain_grad()  # Fixed spelling here
    
    z_det = vae.reparameterize(mu_det, log_var_det).detach() # Detached blocks gradient flow to encoder
    loss_det = vae.decode(z_det).sum()
    loss_det.backward()
    
    # Since mu_det was detached via reparameterization, its gradient will be None or zero
    grad_without_trick = mu_det.grad.norm().item() if mu_det.grad is not None else 0.0
    
    print(f"Check 2 - Encoder grad norm (with trick): {grad_with_trick:.4f} | (detached): {grad_without_trick:.4f}")
    # Check 3: Loss at beta=1 is single-sample negative ELBO
    loss_val = vae_loss_fn(vae, Xtr[:32], beta=1.0)
    print(f"Check 3 - VAE loss at beta=1.0 runs successfully: {loss_val.item():.4f}")

Check 1 - Closed-form KL: 6.5805 | Monte Carlo KL: 6.5815
Check 2 - Encoder grad norm (with trick): 4.6129 | (detached): 0.0000
Check 3 - VAE loss at beta=1.0 runs successfully: 44.5212
